# Libraries

In [ ]:
!pip uninstall -y torch torchvision torchaudio torchao

!pip install -q torch==2.10.0 torchvision==0.25.0 torchaudio==2.10.0 torchao==0.16.0 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
# Install GPTQModel
!pip install -q gptqmodel==5.0.0
!pip install -q optimum

In [ ]:
!pip install -q vllm==0.19.0

In [ ]:
# Remove incompatible packages
!pip uninstall -y \
tensorflow \
google-adk \
google-ai-generativelanguage \
grpcio-status \
google-cloud-aiplatform \
google-cloud-bigtable \
google-cloud-firestore \
google-cloud-discoveryengine \
google-cloud-spanner \
bigframes \
protobuf \
numba-cuda \
opentelemetry-api \
opentelemetry-sdk \
cuda-core

# Install modern protobuf stack
!pip install -q \
protobuf==7.34.0 \
numpy==2.2.6 \
grpcio \
wandb \
opentelemetry-api==1.42.1 \
opentelemetry-sdk==1.42.1 \
numba-cuda==0.22.2 \
cuda-core==0.3.2

In [ ]:
# Standard library imports
import os
import json
import random
import shutil
import subprocess

# Kaggle environment
from kaggle_secrets import UserSecretsClient

# Deep learning framework (must precede vllm)
import torch

# Third-party libraries
import numpy as np
from transformers import AutoTokenizer, set_seed
from huggingface_hub import login, snapshot_download
from vllm import LLM, SamplingParams
from safetensors.torch import load_file, save_file
import traceback

# Global Variables

In [ ]:
# User & Repository Configuration
USER_NAME = "Abdo0213"
EMAIL     = "abdelrahman.ahmed0901@gmail.com"
REPO      = "MahmoudOsama20/EdgeCompress"  # Target repository

# Model Selection
MODEL = "Phi-4-mini-instruct"

# Model & Tokenizer Configuration
MODEL_ID     = "EdgeCompress01/Phi-4-mini-instruct-WANDA-GPTQ-4bit"
TOKENIZER_ID = "microsoft/Phi-4-mini-instruct"

MODEL_NAME           = "Phi-4-mini-instruct"
MODEL_NAME_IN_REPO   = "Phi-4-mini-instruct-WANDA-GPTQ"
COMPRESSION_METHOD   = "WANDA & GPTQ"
MODEL_TYPE           = "Pruning & Quantization"
SPARSITY = 70
PATH = f"Models/{SPARSITY}"

# Inference Prompt (Chat Format)
PROMPT = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Provide a concise explanation of Artificial Intelligence."
    }
]

dummy_prompt = [
    {
        "role": "system",
        "content": "You are a helpful assistant."
    },
    {
        "role": "user",
        "content": "Give me story of a brave man"
    }
]

# Generation Configuration
MAX_GENERATION_TOKENS = 150
SEED = 42

# Output Configuration
OUTPUT_FILE = "model_metrics.json"

# Functions

In [ ]:
def set_reproducibility(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    set_seed(seed)
    
    # Ensure deterministic behavior in CuDNN
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Seeding

In [ ]:
set_reproducibility(SEED)

# Huggingface Login

In [ ]:
# HUGGING FACE AUTHENTICATION
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
login(token=hf_token)
print("Logging into Hugging Face...")

# GitHub login

In [ ]:
token = UserSecretsClient().get_secret("GIT_TOKEN")
print("Logging into GitHub...")

# Download Model

In [ ]:
local_path = snapshot_download(
    repo_id=MODEL_ID,
    allow_patterns=f"{PATH}/*",
    local_dir="/kaggle/working/model"
)

# Prompt Preprocessing

**DummyPrompt Tokenization**

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID)

#Chat Formating
dummy_prompt_text = tokenizer.apply_chat_template(
    dummy_prompt, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
dummy_prompt_token_ids = tokenizer.encode(dummy_prompt_text)

**RealPrompt Tokenization**

In [ ]:
#Chat Formating
prompt_text = tokenizer.apply_chat_template(
    PROMPT, 
    tokenize=False, 
    add_generation_prompt=True
)

#Tokenization
prompt_token_ids = tokenizer.encode(prompt_text)

# Initializing vLLM

In [ ]:

src = f"{local_path}/{PATH}"
dst = "/kaggle/working/model_fused"
os.makedirs(dst, exist_ok=True)

# Copy all non-weight files
for f in os.listdir(src):
    if not f.endswith(".safetensors"):
        shutil.copy2(os.path.join(src, f), os.path.join(dst, f))

weights = load_file(f"{src}/model.safetensors")
new_weights = {}
processed = set()

for key in weights:
    if key in processed:
        continue
    if ".mlp.gate_proj." in key:
        suffix = key.split(".mlp.gate_proj.")[-1]
        prefix = key.split(".mlp.gate_proj.")[0]
        up_key = f"{prefix}.mlp.up_proj.{suffix}"
        fused_key = f"{prefix}.mlp.gate_up_proj.{suffix}"

        gate_t = weights[key]
        up_t   = weights[up_key]

        if suffix == "g_idx":
            # g_idx indexes input groups — same for both, keep one
            new_weights[fused_key] = gate_t
        elif suffix == "qweight":
            # qweight: [in_features//8, out_features] → concat on out dim (dim=1)
            new_weights[fused_key] = torch.cat([gate_t, up_t], dim=1)
        else:
            # scales, qzeros: [n_groups, out_features] → concat on out dim (dim=1)
            new_weights[fused_key] = torch.cat([gate_t, up_t], dim=1)

        processed.update([key, up_key])
    elif ".mlp.up_proj." in key:
        processed.add(key)  # already fused above
    else:
        new_weights[key] = weights[key]
        processed.add(key)

save_file(new_weights, f"{dst}/model.safetensors")
print(f"Done. {len(weights)} → {len(new_weights)} tensors")

In [ ]:
try:
    llm = LLM(
        model=dst,
        tokenizer=TOKENIZER_ID,
        dtype="float16",
        max_model_len=256,
        tensor_parallel_size=2,
        gpu_memory_utilization=0.85,
        attention_backend="TRITON_ATTN",
        disable_log_stats=False,
        enable_prefix_caching=False,
    )
    print("vLLM Initialized Successfully!")
except Exception as e:
    traceback.print_exc()

sampling_params = SamplingParams(
    temperature=0.0,
    max_tokens=MAX_GENERATION_TOKENS,
    min_tokens=MAX_GENERATION_TOKENS,
    seed = SEED
)

# Inference

In [ ]:
# ── Initialize collectors ──────────────────────────────────────────
ttft_times      = []
latency_times   = []
decode_times    = []
inference_times = []  # prefill + decode

# ── Warm-up ONCE, outside the measurement loop ────────────────────
for _ in range(5):
    llm.generate(
        prompts=[{"prompt_token_ids": dummy_prompt_token_ids}],
        sampling_params=SamplingParams(max_tokens=50, temperature=0.0, seed=SEED),
        use_tqdm=False,
    )

# ── Measurement loop ──────────────────────────────────────────────
N_RUNS = 30
for _ in range(N_RUNS):
    output  = llm.generate(
        prompts=[{"prompt_token_ids": prompt_token_ids}],
        sampling_params=sampling_params,
        use_tqdm=False,
    )
    metrics = output[0].metrics

    ttft      = metrics.first_token_ts - metrics.scheduled_ts
    latency   = metrics.last_token_ts  - metrics.queued_ts
    decode    = metrics.last_token_ts  - metrics.first_token_ts
    inference = ttft + decode                                   # prefill + decode

    ttft_times.append(ttft)
    latency_times.append(latency)
    decode_times.append(decode)
    inference_times.append(inference)

# ── Stats ─────────────────────────────────────────────────────────
ttft_arr, latency_arr, decode_arr, inference_arr = (
    np.array(ttft_times),
    np.array(latency_times),
    np.array(decode_times),
    np.array(inference_times),
)

ttft_mean,      ttft_std      = ttft_arr.mean(),      ttft_arr.std()
latency_mean,   latency_std   = latency_arr.mean(),   latency_arr.std()
inference_mean, inference_std = inference_arr.mean(), inference_arr.std()

# decode throughput: excludes first token, over decode-only window
decode_throughput_arr                           = (MAX_GENERATION_TOKENS - 1) / decode_arr
decode_throughput_mean, decode_throughput_std   = (
    decode_throughput_arr.mean(), decode_throughput_arr.std()
)

# overall throughput: all tokens over e2e latency
overall_throughput_arr                          = MAX_GENERATION_TOKENS / latency_arr
overall_throughput_mean, overall_throughput_std = (
    overall_throughput_arr.mean(), overall_throughput_arr.std()
)

input_tokens           = len(prompt_token_ids)
total_generated_tokens = MAX_GENERATION_TOKENS

In [ ]:
print(latency_times)

In [ ]:
print(ttft_times)

# Results

**Writing in json file**

In [ ]:
benchmark_results = {
    "model_name"                        : MODEL_NAME,
    "model_type"                       : MODEL_TYPE,
    "compression_method"               : COMPRESSION_METHOD,
    "sparsity"                         : SPARSITY,
    "input_token_count"                 : input_tokens,
    "generated_token_count"             : total_generated_tokens,
    "ttft_sec"                          : f"{ttft_mean:.2f} ± {ttft_std:.2f}",
    "inference_sec"                     : f"{inference_mean:.2f} ± {inference_std:.2f}",
    "latency_sec"                       : f"{latency_mean:.2f} ± {latency_std:.2f}",
    "decode_throughput_tokens_per_sec"  : f"{decode_throughput_mean:.2f} ± {decode_throughput_std:.2f}",
    "overall_throughput_tokens_per_sec" : f"{overall_throughput_mean:.2f} ± {overall_throughput_std:.2f}",
}

# Save Results
with open(OUTPUT_FILE, "w", encoding="utf-8") as file:
    json.dump(benchmark_results, file, indent=4, ensure_ascii=False)

print(f"[INFO] Metrics successfully saved to '{OUTPUT_FILE}'")

**Push Results to GitHub Repository**

In [ ]:
# Paths & Repository Setup
target_dir_in_repo = f"Evaluations/InferenceEvaluations/CloudEvaluation/{MODEL}/{MODEL_NAME_IN_REPO}/Sparsity/{SPARSITY}"
source_file = OUTPUT_FILE 
remote_url = f"https://{token}@github.com/{REPO}.git"


# Local temporary directory
current_dir = os.getcwd()
temp_repo_dir = os.path.join(current_dir, "temp_git_repo")


# Clean Previous Runs
if os.path.exists(temp_repo_dir):
    shutil.rmtree(temp_repo_dir)


# Clone Repository
print(f"Cloning repository into: {temp_repo_dir}")
subprocess.run(
    ["git", "clone", remote_url, temp_repo_dir],
    check=True
)


# Prepare Target Directory
full_target_path = os.path.join(temp_repo_dir, target_dir_in_repo)
os.makedirs(full_target_path, exist_ok=True)


# Copy Source File (FIXED)
if os.path.exists(source_file):
    dest_path = os.path.join(full_target_path, os.path.basename(source_file))
    shutil.copy2(source_file, dest_path)
else:
    print(f"Warning: Source file '{source_file}' does not exist.")


# Git Config, Commit & Push
try:
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.email", EMAIL],
        check=True
    )
    subprocess.run(
        ["git", "-C", temp_repo_dir, "config", "user.name", USER_NAME],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "add", "."],
        check=True
    )

    commit_msg = f"Added the cloud inference evaluation results to {target_dir_in_repo}"
    subprocess.run(
        ["git", "-C", temp_repo_dir, "commit", "-m", commit_msg],
        check=True
    )

    subprocess.run(
        ["git", "-C", temp_repo_dir, "push", "origin", "main"],
        check=True
    )

    print(f"Successfully pushed to '{REPO}' → '{target_dir_in_repo}'")

except subprocess.CalledProcessError as error:
    print(f"Git operation failed: {error}")